In [1]:
import subprocess, sys, time, json, os, numpy as np
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv; python --version; free -g | head -2; nproc; df -h /kaggle/working | tail -1", shell=True, capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vec2text==0.0.13", "faiss-cpu"], capture_output=True, text=True); print("pip rc", r.returncode, r.stderr[-1200:])
import torch, transformers; print("torch", torch.__version__, "transformers", transformers.__version__, "gpus", torch.cuda.device_count())

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Python 3.13.15
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           5          29
4
/dev/loop2       20G   72K   20G   1% /kaggle/working

pip rc 0 
torch 2.11.0+cu128 transformers 5.16.1 gpus 2


In [2]:
from datasets import load_dataset; from sentence_transformers import SentenceTransformer
passages = list(load_dataset("sentence-transformers/natural-questions", split="train[:3000]")["answer"]); m = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")
lens = [len(x) for x in m.tokenizer(passages[:1000])["input_ids"]]; print("passage tokens mean/median/p95/max", np.mean(lens), np.median(lens), np.percentile(lens, 95), max(lens), "max_seq_length", m.max_seq_length)
res = {}; m.half(); m.encode(passages[:64], batch_size=32); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time(); e = m.encode(passages[:2000], batch_size=32, normalize_embeddings=True); torch.cuda.synchronize(); dt = time.time() - t; res["fp16"] = {"n": 2000, "sec": round(dt, 1), "per_s": round(2000 / dt, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
m.float(); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time(); e = m.encode(passages[:500], batch_size=32, normalize_embeddings=True); torch.cuda.synchronize(); dt = time.time() - t; res["fp32"] = {"n": 500, "sec": round(dt, 1), "per_s": round(500 / dt, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}
print(json.dumps(res)); json.dump(res, open("/kaggle/working/bench_bge.json", "w"))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (555 > 512). Running this sequence through the model will result in indexing errors


passage tokens mean/median/p95/max 135.706 121.0 260.0999999999999 555 max_seq_length 512
{"fp16": {"n": 2000, "sec": 11.7, "per_s": 170.4, "peak_gb": 1.06}, "fp32": {"n": 500, "sec": 15.1, "per_s": 33.0, "peak_gb": 2.1}}


In [1]:
import subprocess, sys, time, json, numpy as np; r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vec2text==0.0.13", "transformers==4.46.3", "tokenizers==0.20.3", "faiss-cpu"], capture_output=True, text=True); print("pip rc", r.returncode, r.stderr[-1500:])
import torch, transformers; torch.set_grad_enabled(False); print("torch", torch.__version__, "transformers", transformers.__version__)
import vec2text; from transformers import AutoModel, AutoTokenizer; from sacrebleu import corpus_bleu; from datasets import load_dataset; from sentence_transformers import SentenceTransformer as ST
passages = list(load_dataset("sentence-transformers/natural-questions", split="train[:200]")["answer"]); corrector = vec2text.load_pretrained_corrector("gtr-base"); enc = AutoModel.from_pretrained("sentence-transformers/gtr-t5-base").encoder.to("cuda").eval(); tk = AutoTokenizer.from_pretrained("sentence-transformers/gtr-t5-base")
orig = [tk.decode(tk(p, max_length=32, truncation=True)["input_ids"], skip_special_tokens=True) for p in passages[:40]]; pool = vec2text.models.model_utils.mean_pool
emb = lambda texts: (lambda x: pool(enc(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).last_hidden_state, x["attention_mask"]))(tk(texts, return_tensors="pt", max_length=32, truncation=True, padding="max_length").to("cuda"))
out_res = []
for n, steps, beam in [(4, 1, 0), (32, 1, 0), (32, 20, 0), (8, 50, 8)]: texts = orig[:n]; e = emb(texts); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time(); out = vec2text.invert_embeddings(embeddings=e, corrector=corrector, num_steps=steps, sequence_beam_width=beam); torch.cuda.synchronize(); dt = time.time() - t; cos = torch.nn.functional.cosine_similarity(e, emb(out)).mean().item(); out_res.append({"n": n, "steps": steps, "beam": beam, "sec": round(dt, 1), "sec_per_sample": round(dt / n, 2), "bleu": round(corpus_bleu(out, [texts]).score, 1), "cos": round(cos, 3), "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}); print(out_res[-1], "| example:", texts[0][:70], "=>", out[0][:70])
st = ST("sentence-transformers/gtr-t5-base", device="cuda"); st.max_seq_length = 32; e_st = st.encode(orig[:32], convert_to_tensor=True); out = vec2text.invert_embeddings(embeddings=e_st, corrector=corrector, num_steps=20, sequence_beam_width=0); print("ST embedding (with dense layer) BLEU", round(corpus_bleu(out, [orig[:32]]).score, 1), "cos(ST, mean-pool)", torch.nn.functional.cosine_similarity(e_st, emb(orig[:32])).mean().item())
json.dump(out_res, open("/kaggle/working/bench_v2t.json", "w"))

pip rc 0 ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.

torch 2.11.0+cu128 transformers 4.46.3


README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/44.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

... (29 outputs left out)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/69.0 [00:00<?, ?B/s]

{'n': 4, 'steps': 1, 'beam': 0, 'sec': 1.7, 'sec_per_sample': 0.43, 'bleu': 42.3, 'cos': 0.971, 'peak_gb': 3.13} | example: Richmond Football Club Richmond began 2017 with 5 straight wins, a fea => Richmond Football Club began 2017 with 5 straight wins. The losses cap
{'n': 32, 'steps': 1, 'beam': 0, 'sec': 1.4, 'sec_per_sample': 0.05, 'bleu': 60.4, 'cos': 0.968, 'peak_gb': 3.84} | example: Richmond Football Club Richmond began 2017 with 5 straight wins, a fea => Richmond Football Club began 2017 with 5 straight wins. The losses cap
{'n': 32, 'steps': 20, 'beam': 0, 'sec': 17.1, 'sec_per_sample': 0.54, 'bleu': 91.2, 'cos': 0.997, 'peak_gb': 3.84} | example: Richmond Football Club Richmond began 2017 with 5 straight wins, a fea => Richmond Football Club Richmond began 2017 with 5 straight wins, a fea


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 574.81 MiB is free. Including non-PyTorch memory, this process has 14.00 GiB memory in use. Of the allocated memory 13.05 GiB is allocated by PyTorch, and 843.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [2]:
import gc; gc.collect(); torch.cuda.empty_cache(); out_res2 = []
st = ST("sentence-transformers/gtr-t5-base", device="cuda"); st.max_seq_length = 32; e_st = st.encode(orig[:32], convert_to_tensor=True); out = vec2text.invert_embeddings(embeddings=e_st, corrector=corrector, num_steps=20, sequence_beam_width=0); print("ST embedding (with dense layer) BLEU", round(corpus_bleu(out, [orig[:32]]).score, 1), "cos(ST, mean-pool)", torch.nn.functional.cosine_similarity(e_st, emb(orig[:32])).mean().item())
for n, steps, beam in [(32, 50, 0), (1, 50, 8), (2, 50, 8)]: texts = orig[:n]; e = emb(texts); torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time(); out = vec2text.invert_embeddings(embeddings=e, corrector=corrector, num_steps=steps, sequence_beam_width=beam); torch.cuda.synchronize(); dt = time.time() - t; cos = torch.nn.functional.cosine_similarity(e, emb(out)).mean().item(); out_res2.append({"n": n, "steps": steps, "beam": beam, "sec": round(dt, 1), "sec_per_sample": round(dt / n, 2), "bleu": round(corpus_bleu(out, [texts]).score, 1), "cos": round(cos, 3), "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)}); print(out_res2[-1])

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

ST embedding (with dense layer) BLEU 0.3 cos(ST, mean-pool) -0.020665377378463745
{'n': 32, 'steps': 50, 'beam': 0, 'sec': 43.6, 'sec_per_sample': 1.36, 'bleu': 91.8, 'cos': 0.997, 'peak_gb': 13.11}
{'n': 1, 'steps': 50, 'beam': 8, 'sec': 69.9, 'sec_per_sample': 69.88, 'bleu': 100.0, 'cos': 1.0, 'peak_gb': 13.95}


OutOfMemoryError: CUDA out of memory. Tried to allocate 518.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 314.81 MiB is free. Including non-PyTorch memory, this process has 14.25 GiB memory in use. Of the allocated memory 13.50 GiB is allocated by PyTorch, and 638.25 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)